In [32]:
import os
import pandas as pd

# Rutas base
base_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN"
spei_dir = r"M:\material_isa\estadisticos\anual\sequia\3_12"

# Iterar sobre todos los archivos filtrados por nieve
for file in os.listdir(base_dir):
    if file.endswith("_sin_periodo_nieve_filtr.csv"):
        nombre_turbera = file.replace("_sin_periodo_nieve_filtr.csv", "").replace("_", "")
        
        # Construir rutas
        input_file = os.path.join(base_dir, file)
        spei_file = os.path.join(spei_dir, f"Sequia_{nombre_turbera}_3_12.csv")
        output_file = os.path.join(base_dir, f"{file.replace('.csv', '_3_12.csv')}")

        try:
            # Leer archivos
            df_filtrado = pd.read_csv(input_file, sep=";", encoding="latin1")
            df_spei = pd.read_csv(spei_file, sep=";")

            # Asegurar nombres de columna
            df_filtrado["FECHA"] = pd.to_datetime(df_filtrado["FECHA"], errors="coerce")
            df_spei["fecha"] = pd.to_datetime(df_spei["fecha"], errors="coerce")

            # Hacer merge por fecha
            df_merged = pd.merge(df_filtrado, df_spei, left_on="FECHA", right_on="fecha", how="left")
            df_merged.drop(columns=["fecha"], inplace=True)

            # Guardar archivo
            df_merged.to_csv(output_file, index=False, sep=";", encoding="latin1")
            print(f"✅ Guardado: {output_file}")

        except Exception as e:
            print(f"❌ Error con {file}: {e}")


✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Estanyeres_sin_periodo_nieve_filtr_3_12.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv


In [34]:
import os
import pandas as pd
import numpy as np

# Rutas de entrada/salida
dir_entrada = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN"
dir_salida = os.path.join(dir_entrada, "Base de datos_Indices_filtrada")
ruta_umbral = os.path.join(dir_entrada, "porcentaje_de_datos_min_a_usar_v2.csv")

# Crear carpeta de salida si no existe
os.makedirs(dir_salida, exist_ok=True)

# Leer tabla de umbrales
df_umbral = pd.read_csv(ruta_umbral, sep=";")
df_umbral.columns = df_umbral.columns.str.strip().str.upper()

# Crear diccionario de umbrales para NDWI
umbrales_ndwi = dict(zip(df_umbral["NOMBRE"], df_umbral["NDATOS_MIN_NDWI"]))

# Procesar cada archivo CSV de entrada
for archivo in os.listdir(dir_entrada):
    if archivo.endswith("_sin_periodo_nieve_filtr_3_12.csv"):
        try:
            ruta_csv = os.path.join(dir_entrada, archivo)
            df = pd.read_csv(ruta_csv, sep=";", encoding="latin1")

            # Nombre de turbera (sin espacios)
            nombre_turbera = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "").replace("_", " ")

            # Obtener umbral de NDWI
            umbral_ndwi = umbrales_ndwi.get(nombre_turbera)
            if umbral_ndwi is None:
                print(f"⚠️ Umbral NDWI no encontrado para {nombre_turbera}")
                continue

            # Aplicar filtro a NDWI en base a NDATOS.2
            if "NDATOS.2" in df.columns and "NDWI" in df.columns:
                df.loc[df["NDATOS.2"] < umbral_ndwi, "NDWI"] = np.nan

            # Guardar archivo resultante
            ruta_salida = os.path.join(dir_salida, archivo)
            df.to_csv(ruta_salida, sep=";", index=False, encoding="latin1")
            print(f"✅ Filtrado y guardado: {archivo}")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

print("\n🎯 Proceso finalizado.")


✅ Filtrado y guardado: Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv
✅ Filtrado y guardado: Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv
✅ Filtrado y guardado: Estanyeres_sin_periodo_nieve_filtr_3_12.csv
✅ Filtrado y guardado: Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv
✅ Filtrado y guardado: Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv
✅ Filtrado y guardado: Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv

🎯 Proceso finalizado.


In [38]:
#-----------------------------
#recorre todos los CSV en el directorio Base de datos_Indices_filtrada, elimina las columnas "Elevación", "Pendiente" y 
#"Orientación" (si existen), y guarda los archivos sobrescribiéndolos
#--------------------------------

import os
import pandas as pd

# Carpeta de entrada
dir_csv = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"

# Columnas a eliminar
columnas_a_eliminar = ["Elevación", "Pendiente", "Orientación"]

# Procesar cada archivo CSV
for archivo in os.listdir(dir_csv):
    if archivo.endswith(".csv"):
        ruta_archivo = os.path.join(dir_csv, archivo)
        try:
            # Leer archivo
            df = pd.read_csv(ruta_archivo, sep=";", encoding="latin1")
            
            # Eliminar columnas si existen
            columnas_presentes = [col for col in columnas_a_eliminar if col in df.columns]
            if columnas_presentes:
                df.drop(columns=columnas_presentes, inplace=True)
                df.to_csv(ruta_archivo, sep=";", index=False, encoding="latin1")
                print(f"✅ Procesado: {archivo} (columnas eliminadas: {', '.join(columnas_presentes)})")
            else:
                print(f"ℹ️ Sin columnas a eliminar en: {archivo}")

        except Exception as e:
            print(f"❌ Error con {archivo}: {e}")

print("\n🎯 Proceso finalizado.")



✅ Procesado: Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv (columnas eliminadas: Elevación, Pendiente, Orientación)
ℹ️ Sin columnas a eliminar en: Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv
ℹ️ Sin columnas a eliminar en: Estanyeres_sin_periodo_nieve_filtr_3_12.csv
ℹ️ Sin columnas a eliminar en: Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv
ℹ️ Sin columnas a eliminar en: Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv
ℹ️ Sin columnas a eliminar en: Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv

🎯 Proceso finalizado.


In [40]:
#--------------
# encabezado en mayusculas
#---------------

import os
import pandas as pd

# Carpeta donde están los archivos
dir_csv = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"

# Procesar cada archivo CSV
for archivo in os.listdir(dir_csv):
    if archivo.endswith(".csv"):
        ruta_archivo = os.path.join(dir_csv, archivo)
        try:
            df = pd.read_csv(ruta_archivo, sep=";", encoding="latin1")

            # Renombrar columnas específicas y convertir todas a mayúsculas
            df.rename(columns={"spei_3": "SPEI_3", "spei_12": "SPEI_12"}, inplace=True)
            df.columns = [col.upper() for col in df.columns]

            # Guardar el archivo sobrescribiendo
            df.to_csv(ruta_archivo, sep=";", index=False, encoding="latin1")
            print(f"✅ Encabezado actualizado: {archivo}")

        except Exception as e:
            print(f"❌ Error en {archivo}: {e}")

print("\n🎯 Proceso finalizado.")




✅ Encabezado actualizado: Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv
✅ Encabezado actualizado: Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv
✅ Encabezado actualizado: Estanyeres_sin_periodo_nieve_filtr_3_12.csv
✅ Encabezado actualizado: Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv
✅ Encabezado actualizado: Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv
✅ Encabezado actualizado: Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv

🎯 Proceso finalizado.


In [50]:
import os
import pandas as pd
import numpy as np

# Carpeta con archivos
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"

# Procesar todos los CSV
for archivo in os.listdir(input_dir):
    if archivo.endswith(".csv"):
        ruta = os.path.join(input_dir, archivo)
        try:
            df = pd.read_csv(ruta, sep=";", encoding="latin1")

            # Verificar que existen las columnas necesarias
            if "NDVI" in df.columns and "NDWI_V" in df.columns:
                # Calcular NDDI evitando divisiones por cero
                df["NDDI"] = np.where(
                    (df["NDVI"] + df["NDWI_V"]) != 0,
                    (df["NDVI"] - df["NDWI_V"]) / (df["NDVI"] + df["NDWI_V"]),
                    np.nan
                )

                # Sobrescribir el archivo con la nueva columna
                df.to_csv(ruta, sep=";", index=False, encoding="latin1")
                print(f"✅ NDDI agregado: {archivo}")
            else:
                print(f"⚠️ NDVI o NDWI no encontrados en: {archivo}")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

print("\n🎯 Cálculo de NDDI completado.")


✅ NDDI agregado: Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv
✅ NDDI agregado: Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv
✅ NDDI agregado: Estanyeres_sin_periodo_nieve_filtr_3_12.csv
✅ NDDI agregado: Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv
✅ NDDI agregado: Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv
✅ NDDI agregado: Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv

🎯 Cálculo de NDDI completado.


In [52]:
import os
import pandas as pd
import numpy as np

# Directorio de entrada y salida
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
output_dir = os.path.join(input_dir, "NDDI_filtr")

# Crear carpeta de salida si no existe
os.makedirs(output_dir, exist_ok=True)

# Recorrer los archivos CSV
for archivo in os.listdir(input_dir):
    if archivo.endswith(".csv"):
        ruta_entrada = os.path.join(input_dir, archivo)
        ruta_salida = os.path.join(output_dir, archivo)

        try:
            df = pd.read_csv(ruta_entrada, sep=";", encoding="latin1")

            if "NDDI" in df.columns:
                fuera_de_rango = (~df["NDDI"].between(-1, 1)).sum()
                df["NDDI"] = df["NDDI"].where(df["NDDI"].between(-1, 1), np.nan)

                # Guardar archivo filtrado
                df.to_csv(ruta_salida, sep=";", index=False, encoding="latin1")
                print(f"✅ Filtrado '{archivo}': {fuera_de_rango} valores fuera de [-1, 1] reemplazados por NaN.")
            else:
                print(f"⚠️ 'NDDI' no encontrado en: {archivo}")

        except Exception as e:
            print(f"❌ Error procesando '{archivo}': {e}")

print("\n🎯 Todos los archivos procesados y guardados en carpeta: NDDI_filtr")



✅ Filtrado 'Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv': 389 valores fuera de [-1, 1] reemplazados por NaN.
✅ Filtrado 'Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv': 343 valores fuera de [-1, 1] reemplazados por NaN.
✅ Filtrado 'Estanyeres_sin_periodo_nieve_filtr_3_12.csv': 359 valores fuera de [-1, 1] reemplazados por NaN.
✅ Filtrado 'Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv': 349 valores fuera de [-1, 1] reemplazados por NaN.
✅ Filtrado 'Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv': 415 valores fuera de [-1, 1] reemplazados por NaN.
✅ Filtrado 'Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv': 463 valores fuera de [-1, 1] reemplazados por NaN.

🎯 Todos los archivos procesados y guardados en carpeta: NDDI_filtr


In [66]:
# ----------------------------------------
# GRAFICOS COMPARACION ANTES DESPUES DE FILTRAR... SOLO PARA NDWI
# ----------------------------------------

import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re

# -------------------------------
# CONFIGURACIÓN
# -------------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
dir_original = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
dir_salida_comparativo = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues"
snow_txt_path = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"

limites_y = {
    'NDVI': (0.0, 1.0),
    'NDWI': (-0.3, 1.0),
    'NDWI_v': (0.0, 1.0),
    'NDSI': (-0.2, 1.0)
}

# -------------------------------
# FUNCIONES AUXILIARES
# -------------------------------

def cargar_periodos_nieve(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    torberas_periodos = {}
    bloques = contenido.split('✅ Procesado correctamente: ')[1:]
    for bloque in bloques:
        lineas = bloque.split('\n')
        nombre_raw = lineas[0].strip().replace('.csv', '')
        nombre = nombre_raw.replace('_periodo_nieve', '')
        periodos = []
        for linea in lineas:
            match = re.search(r'Snow in: (\d{2}/\d{2}/\d{4}) \| Snow off: (\d{2}/\d{2}/\d{4})', linea)
            if match:
                snow_in = datetime.strptime(match.group(1), '%d/%m/%Y')
                snow_off = datetime.strptime(match.group(2), '%d/%m/%Y')
                periodos.append((snow_in, snow_off))
        torberas_periodos[nombre] = periodos
    return torberas_periodos

def detectar_formato_fecha(fecha_str):
    if '/' in fecha_str:
        return "%d/%m/%Y"
    elif '-' in fecha_str:
        return "%Y-%m-%d"
    else:
        return None

# -------------------------------
# BLOQUE PRINCIPAL
# -------------------------------

print("\n🔍 Iniciando comparativas antes vs después del filtrado...")

periodos_nieve = cargar_periodos_nieve(snow_txt_path)

for file in os.listdir(input_dir):
    if file.endswith("_sin_periodo_nieve_filtr_3_12.csv"):
        nombre = file.replace("_sin_periodo_nieve_filtr_3_12.csv", "")
        ruta_filtrado = os.path.join(input_dir, file)
        ruta_original = os.path.join(dir_original, f"{nombre}_sin_periodo_nieve.csv")

        try:
            df_filtrado = pd.read_csv(ruta_filtrado, sep=";", encoding="latin1")
            df_original = pd.read_csv(ruta_original, sep=";", encoding="latin1")
        except Exception as e:
            print(f"❌ Error al cargar archivos para {nombre}: {e}")
            continue

        for indice in ['NDWI']:
            if indice in df_filtrado.columns and indice in df_original.columns:
                try:
                    df_filtrado['FECHA'] = df_filtrado['FECHA'].astype(str).str.strip()
                    df_original['FECHA'] = df_original['FECHA'].astype(str).str.strip()

                    # Detectar formatos individuales
                    formato_filt = detectar_formato_fecha(df_filtrado['FECHA'].dropna().iloc[0])
                    formato_orig = detectar_formato_fecha(df_original['FECHA'].dropna().iloc[0])

                    df_filtrado['FECHA_dt'] = pd.to_datetime(df_filtrado['FECHA'], format=formato_filt, errors='coerce')
                    df_original['FECHA_dt'] = pd.to_datetime(df_original['FECHA'], format=formato_orig, errors='coerce')

                    df_filtrado = df_filtrado.dropna(subset=['FECHA_dt'])
                    df_original = df_original.dropna(subset=['FECHA_dt'])

                    df_filtrado = df_filtrado[df_filtrado[indice].astype(str).str.strip() != ""]
                    df_original = df_original[df_original[indice].astype(str).str.strip() != ""]

                    if df_filtrado.empty or df_original.empty:
                        print(f"⚠️ Sin datos válidos para {indice} en {nombre}...")
                        continue

                    fechas_filt = df_filtrado['FECHA_dt']
                    fechas_orig = df_original['FECHA_dt']
                    valores_filt = df_filtrado[indice]
                    valores_orig = df_original[indice]

                    xmin = min(fechas_filt.min(), fechas_orig.min())
                    xmax = max(fechas_filt.max(), fechas_orig.max())

                    plt.figure(figsize=(10, 5))
                    plt.scatter(fechas_orig, valores_orig, color='red', s=12, label='Filtrado')
                    plt.scatter(fechas_filt, valores_filt, color='blue', s=12, label='Original')
                    plt.title(f"Filtrado de {indice} – {nombre.replace('_', ' ')}", fontsize=12, pad=20)
                    plt.xlabel("Fecha")
                    plt.ylabel(indice)
                    plt.grid(True)
                    plt.ylim(*limites_y.get(indice, (-1, 1)))
                    plt.xlim(xmin, xmax)
                    plt.legend()

                    for snow_in, snow_off in periodos_nieve.get(nombre, []):
                        plt.axvspan(snow_in, snow_off, color='gray', alpha=0.2)

                    plt.tight_layout()
                    os.makedirs(dir_salida_comparativo, exist_ok=True)
                    salida_path = os.path.join(dir_salida_comparativo, f"{indice}_{nombre}_comparado.png")
                    plt.savefig(salida_path, dpi=150)
                    plt.close()
                    print(f"📌 Comparativo guardado: {salida_path}")

                except Exception as e:
                    print(f"❌ Error graficando {indice} para {nombre}: {e}")

print("\n🎯 Comparativas finalizadas.")



🔍 Iniciando comparativas antes vs después del filtrado...
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Bassa_Nera_comparado.png
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Clots_de_Rialba_comparado.png
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Estanyeres_comparado.png
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Plan_deth_Horn_comparado.png
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Pla_de_Sartari_comparado.png
📌 Comparativo guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Antes_despues\NDWI_Pleta_de_Molieres_comparado.png

🎯 Comparativas finalizadas.


In [76]:
# ----------------------------------------
# GRAFICOS series temporales. todas las turberas, todos los indices (sequia tambien) con rangos de SNOW
# ----------------------------------------

import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re

# -------------------------------
# CONFIGURACIÓN
# -------------------------------

input_dir_indices = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
input_dir_nddi = os.path.join(input_dir_indices, "NDDI_filtr")
snow_txt_path = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"

limites_y = {
    'NDVI': (0.0, 1.0),
    'NDWI': (-0.3, 1.0),
    'NDWI_V': (0.0, 1.0),
    'NDDI': (-1.0, 1.0),
    'SPEI_3': (-3.0, 3.0),
    'SPEI_12':(-2.0, 2.0)
}

# -------------------------------
# FUNCIONES
# -------------------------------

def cargar_periodos_nieve(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    torberas_periodos = {}
    bloques = contenido.split('✅ Procesado correctamente: ')[1:]
    for bloque in bloques:
        lineas = bloque.split('\n')
        nombre_raw = lineas[0].strip().replace('.csv', '')
        nombre = nombre_raw.replace('_periodo_nieve', '')
        periodos = []
        for linea in lineas:
            match = re.search(r'Snow in: (\d{2}/\d{2}/\d{4}) \| Snow off: (\d{2}/\d{2}/\d{4})', linea)
            if match:
                snow_in = datetime.strptime(match.group(1), '%d/%m/%Y')
                snow_off = datetime.strptime(match.group(2), '%d/%m/%Y')
                periodos.append((snow_in, snow_off))
        torberas_periodos[nombre] = periodos
    return torberas_periodos

def detectar_formato_fecha(fecha_str):
    if '/' in fecha_str:
        return "%d/%m/%Y"
    elif '-' in fecha_str:
        return "%Y-%m-%d"
    else:
        return None

def graficar_indice(df, indice, nombre, periodos, out_path):
    df = df.copy()
    df['FECHA'] = df['FECHA'].astype(str).str.strip()
    formato_fecha = detectar_formato_fecha(df['FECHA'].dropna().iloc[0])
    df['FECHA_dt'] = pd.to_datetime(df['FECHA'], format=formato_fecha, errors='coerce')
    df = df.dropna(subset=['FECHA_dt'])

    if df.empty or indice not in df.columns:
        print(f"⚠️ No hay datos válidos para {indice} en {nombre}")
        return

    fechas = df['FECHA_dt']
    valores = df[indice]
    xmin, xmax = fechas.min(), fechas.max()

    plt.figure(figsize=(10, 5))
    plt.scatter(fechas, valores, s=12)
    plt.title(f"{indice} MEDIANA – TOT\n{nombre.replace('_', ' ')}", fontsize=12, pad=20)
    plt.xlabel("Fecha")
    plt.ylabel(indice)
    plt.grid(True)
    plt.ylim(*limites_y.get(indice, (-1, 1)))
    plt.xlim(xmin, xmax)

    for snow_in, snow_off in periodos:
        plt.axvspan(snow_in, snow_off, color='gray', alpha=0.2)

    plt.text(1.01, 1.02, "Período SNOW", transform=plt.gca().transAxes,
             fontsize=9, ha='right', color='gray',
             bbox=dict(boxstyle="round,pad=0.2", facecolor='lightgray', edgecolor='none', alpha=0.5))

    plt.tight_layout()
    os.makedirs(out_path, exist_ok=True)
    path_final = os.path.join(out_path, f"{indice}_{nombre}.png")
    plt.savefig(path_final, dpi=150)
    plt.close()
    print(f"✅ Guardado: {path_final}")

# -------------------------------
# EJECUCIÓN
# -------------------------------

periodos_dict = cargar_periodos_nieve(snow_txt_path)

for archivo in os.listdir(input_dir_indices):
    if archivo.endswith("_sin_periodo_nieve_filtr_3_12.csv"):
        nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")

        ruta = os.path.join(input_dir_indices, archivo)
        try:
            df = pd.read_csv(ruta, sep=";", encoding="latin1")
        except Exception as e:
            print(f"❌ Error al leer {archivo}: {e}")
            continue

        periodos = periodos_dict.get(nombre, [])

        for indice in ['NDVI', 'NDWI', 'NDWI_V', 'SPEI_3', 'SPEI_12']:
            if indice in df.columns:
                salida_dir = os.path.join(input_dir_indices, indice)
                graficar_indice(df, indice, nombre, periodos, salida_dir)

        # NDDI (ruta diferente)
        ruta_nddi = os.path.join(input_dir_nddi, archivo)
        if os.path.exists(ruta_nddi):
            try:
                df_nddi = pd.read_csv(ruta_nddi, sep=";", encoding="latin1")
                graficar_indice(df_nddi, "NDDI", nombre, periodos, input_dir_nddi)
            except Exception as e:
                print(f"❌ Error con NDDI en {nombre}: {e}")

print("\n🎯 Todos los gráficos de series temporales generados.")


✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDVI\NDVI_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDWI\NDWI_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDWI_V\NDWI_V_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\SPEI_3\SPEI_3_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\SPEI_12\SPEI_12_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr\NDDI_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDVI\NDVI_Clots_de_Rialba.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_

In [74]:
# ----------------------------------------
# EXPERIMENTO reescalar NDDI en lugar de filtrar valores extremos
# ----------------------------------------


import os
import pandas as pd
import numpy as np

# Directorio de entrada y salida
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
output_dir = os.path.join(input_dir, "NDDI_reescalado")

# Crear carpeta de salida si no existe
os.makedirs(output_dir, exist_ok=True)

# Recorrer los archivos CSV
for archivo in os.listdir(input_dir):
    if archivo.endswith(".csv"):
        ruta_entrada = os.path.join(input_dir, archivo)
        ruta_salida = os.path.join(output_dir, archivo)

        try:
            df = pd.read_csv(ruta_entrada, sep=";", encoding="latin1")

            if "NDDI" in df.columns:
                valores_validos = df["NDDI"].dropna()
                if not valores_validos.empty:
                    min_val = valores_validos.min()
                    max_val = valores_validos.max()

                    if max_val != min_val:
                        df["NDDI"] = 2 * (df["NDDI"] - min_val) / (max_val - min_val) - 1
                        print(f"✅ Reescalado '{archivo}' al rango [-1, 1]")
                    else:
                        print(f"⚠️ Sin rango para reescalar en '{archivo}' (todos los valores iguales)")
                else:
                    print(f"⚠️ Sin valores válidos de NDDI en '{archivo}'")
                
                # Guardar archivo reescalado
                df.to_csv(ruta_salida, sep=";", index=False, encoding="latin1")

            else:
                print(f"⚠️ 'NDDI' no encontrado en: {archivo}")

        except Exception as e:
            print(f"❌ Error procesando '{archivo}': {e}")

print("\n🎯 Todos los archivos procesados y guardados en carpeta: NDDI_reescalado")


✅ Reescalado 'Bassa_Nera_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]
✅ Reescalado 'Clots_de_Rialba_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]
✅ Reescalado 'Estanyeres_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]
✅ Reescalado 'Plan_deth_Horn_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]
✅ Reescalado 'Pla_de_Sartari_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]
✅ Reescalado 'Pleta_de_Molieres_sin_periodo_nieve_filtr_3_12.csv' al rango [-1, 1]

🎯 Todos los archivos procesados y guardados en carpeta: NDDI_reescalado


In [78]:
# ----------------------------------------
# EXPERIMENTO comparar NDDI filtrado y escalado
# ----------------------------------------


import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# CONFIGURACIÓN
# -------------------------------
input_filtrado = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr"
input_reescalado = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_reescalado"
salida_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado"

os.makedirs(salida_dir, exist_ok=True)

# -------------------------------
# COMPARACIÓN PARA CADA ARCHIVO
# -------------------------------
for archivo in os.listdir(input_filtrado):
    if archivo.endswith(".csv"):
        ruta_filtrado = os.path.join(input_filtrado, archivo)
        ruta_reescalado = os.path.join(input_reescalado, archivo)
        nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")

        if not os.path.exists(ruta_reescalado):
            print(f"⚠️ Reescalado no encontrado para: {archivo}")
            continue

        try:
            df_f = pd.read_csv(ruta_filtrado, sep=";", encoding="latin1")
            df_r = pd.read_csv(ruta_reescalado, sep=";", encoding="latin1")

            # Parsear fechas
            df_f['FECHA'] = pd.to_datetime(df_f['FECHA'], errors='coerce')
            df_r['FECHA'] = pd.to_datetime(df_r['FECHA'], errors='coerce')

            df_f = df_f.dropna(subset=['FECHA', 'NDDI'])
            df_r = df_r.dropna(subset=['FECHA', 'NDDI'])

            if df_f.empty or df_r.empty:
                print(f"⚠️ Sin datos válidos para: {archivo}")
                continue

            # -------------------------------
            # PLOTEO
            # -------------------------------
            plt.figure(figsize=(10, 5))
            plt.scatter(df_f['FECHA'], df_f['NDDI'], s=30, facecolors='none', edgecolors='blue', label='Filtrado')
            plt.scatter(df_r['FECHA'], df_r['NDDI'], s=30, facecolors='none', edgecolors='red', label='Reescalado')

            plt.title(f"Comparación NDDI – {nombre.replace('_', ' ')}", fontsize=13)
            plt.xlabel("Fecha")
            plt.ylabel("NDDI")
            plt.grid(True)
            plt.legend()
            plt.ylim(-1.2, 1.2)

            plt.tight_layout()
            salida_path = os.path.join(salida_dir, f"NDDI_comparado_{nombre}.png")
            plt.savefig(salida_path, dpi=150)
            plt.close()
            print(f"✅ Guardado: {salida_path}")

        except Exception as e:
            print(f"❌ Error con {archivo}: {e}")

print("\n🎯 Comparaciones completadas.")


✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Bassa_Nera.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Clots_de_Rialba.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Estanyeres.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Plan_deth_Horn.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Pla_de_Sartari.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr_vs_reescalado\NDDI_comparado_Pleta_de_Molieres.png

🎯 Comparacion

In [92]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re

# -------------------------------
# CONFIGURACIÓN
# -------------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
input_dir_nddi = os.path.join(input_dir, "NDDI_filtr")
output_dir = os.path.join(input_dir, "Por_turbera")
snow_txt_path = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"

os.makedirs(output_dir, exist_ok=True)

# -------------------------------
# FUNCIONES
# -------------------------------

def cargar_periodos_nieve(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    torberas_periodos = {}
    bloques = contenido.split('✅ Procesado correctamente: ')[1:]
    for bloque in bloques:
        lineas = bloque.split('\n')
        nombre_raw = lineas[0].strip().replace('.csv', '')
        nombre = nombre_raw.replace('_periodo_nieve', '')
        periodos = []
        for linea in lineas:
            match = re.search(r'Snow in: (\d{2}/\d{2}/\d{4}) \| Snow off: (\d{2}/\d{2}/\d{4})', linea)
            if match:
                snow_in = datetime.strptime(match.group(1), '%d/%m/%Y')
                snow_off = datetime.strptime(match.group(2), '%d/%m/%Y')
                periodos.append((snow_in, snow_off))
        torberas_periodos[nombre] = periodos
    return torberas_periodos

# -------------------------------
# EJECUCIÓN
# -------------------------------
periodos_nieve = cargar_periodos_nieve(snow_txt_path)

for archivo in os.listdir(input_dir):
    if not archivo.endswith(".csv") or "NDDI" in archivo:
        continue

    nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")
    ruta_base = os.path.join(input_dir, archivo)
    ruta_nddi = os.path.join(input_dir_nddi, archivo)

    try:
        df = pd.read_csv(ruta_base, sep=";", encoding="latin1")
        df_nddi = pd.read_csv(ruta_nddi, sep=";", encoding="latin1") if os.path.exists(ruta_nddi) else pd.DataFrame()

        df['FECHA'] = df['FECHA'].astype(str).str.strip()
        df['FECHA_dt'] = pd.to_datetime(df['FECHA'], format=date_format, errors='coerce')
        if not df_nddi.empty:
            df_nddi['FECHA'] = df_nddi['FECHA'].astype(str).str.strip()
            df_nddi['FECHA_dt'] = pd.to_datetime(df_nddi['FECHA'], format=date_format, errors='coerce')

        plt.figure(figsize=(12, 6))

        colores = {
            'NDVI': 'green',
            'NDWI_V': 'blue',
            'NDDI': 'orange',
            'SPEI_3': 'red',
            'SPEI_12': 'black'
        }

        marcadores = {
            'NDVI': 'o',
            'NDWI_V': 'o',
            'NDDI': 'o',
            'SPEI_3': 's',
            'SPEI_12': 's'
        }

        for indice in ['NDVI', 'NDWI_V', 'SPEI_3', 'SPEI_12']:
            if indice in df.columns:
                df_valid = df.dropna(subset=['FECHA_dt', indice])
                plt.scatter(df_valid['FECHA_dt'], df_valid[indice],
                            facecolors='none',
                            edgecolors=colores[indice],
                            label=indice,
                            marker=marcadores[indice],
                            s=5 if 'SPEI' in indice else 20,         # <-- tamaño condicional
                            alpha=0.7 if 'SPEI' in indice else 1.0)    # <-- transparencia condicional
        
        if not df_nddi.empty and 'NDDI' in df_nddi.columns:
            plt.scatter(df_nddi['FECHA_dt'], df_nddi['NDDI'], facecolors='none', edgecolors=colores['NDDI'], label='NDDI', marker='o', s=20)

        plt.xlim(df['FECHA_dt'].min(), df['FECHA_dt'].max())
        plt.grid(True)
        plt.xlabel("Fecha")
        plt.ylabel("Valor del índice")
        plt.title(f"Series Temporales – {nombre.replace('_', ' ')}")
        plt.legend()

        for snow_in, snow_off in periodos_nieve.get(nombre, []):
            plt.axvspan(snow_in, snow_off, color='gray', alpha=0.2)
        
        plt.text(1.01, 1.02, "Período SNOW", transform=plt.gca().transAxes,
             fontsize=9, ha='right', color='gray',
             bbox=dict(boxstyle="round,pad=0.2", facecolor='lightgray', edgecolor='none', alpha=0.5))
        
        plt.tight_layout()
        salida_path = os.path.join(output_dir, f"{nombre}_series_temporales.png")
        plt.savefig(salida_path, dpi=150)
        plt.close()
        print(f"✅ Gráfico generado: {salida_path}")

    except Exception as e:
        print(f"❌ Error procesando {archivo}: {e}")


✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Bassa_Nera_series_temporales.png
✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Clots_de_Rialba_series_temporales.png
✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Estanyeres_series_temporales.png
✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Plan_deth_Horn_series_temporales.png
✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Pla_de_Sartari_series_temporales.png
✅ Gráfico generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Pleta_de_Molieres_series_temporales.png


In [98]:
#----------------------------------------------------
# boxplots de NDDI agrupado por rangos de SPEI
#----------------------------------------------------


import os
import pandas as pd
import matplotlib.pyplot as plt

# Rutas
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr"
output_dir = r"M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots"

os.makedirs(output_dir, exist_ok=True)

# Función para clasificar SPEI
def clasificar_spei(valor):
    if pd.isna(valor):
        return None
    elif valor < -1:
        return "SPEI < -1"
    elif -1 <= valor < 0:
        return "-1 ≤ SPEI < 0"
    elif 0 <= valor < 1:
        return "0 ≤ SPEI < 1"
    elif valor >= 1:
        return "SPEI ≥ 1"

# Procesar archivos
for archivo in os.listdir(input_dir):
    if archivo.endswith(".csv"):
        nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")
        ruta = os.path.join(input_dir, archivo)
        try:
            df = pd.read_csv(ruta, sep=";", encoding="latin1")

            if "NDDI" not in df.columns or ("SPEI_3" not in df.columns and "SPEI_12" not in df.columns):
                print(f"⚠️ Saltado: {archivo} - faltan columnas necesarias")
                continue

            for spei_col in ["SPEI_3", "SPEI_12"]:
                if spei_col not in df.columns:
                    continue

                df_tmp = df[[spei_col, "NDDI"]].dropna()
                df_tmp["Rango_SPEI"] = df_tmp[spei_col].apply(clasificar_spei)
                df_tmp = df_tmp.dropna(subset=["Rango_SPEI"])

                if df_tmp.empty:
                    print(f"⚠️ Sin datos válidos para {spei_col} en {nombre}")
                    continue

                # Gráfico boxplot
                plt.figure(figsize=(8, 5))
                df_tmp.boxplot(column="NDDI", by="Rango_SPEI", grid=True, patch_artist=True)
                plt.title(f"NDDI vs {spei_col} – {nombre.replace('_', ' ')}")
                plt.suptitle("")  # Quitar supertítulo
                plt.xlabel("Rango de " + spei_col)
                plt.ylabel("NDDI")
                plt.tight_layout()

                out_path = os.path.join(output_dir, f"boxplot_{nombre}_{spei_col}.png")
                plt.savefig(out_path, dpi=150)
                plt.close()
                print(f"📦 Guardado: {out_path}")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

print("\n🎯 Boxplots completados.")


📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Bassa_Nera_SPEI_3.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Bassa_Nera_SPEI_12.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Clots_de_Rialba_SPEI_3.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Clots_de_Rialba_SPEI_12.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Estanyeres_SPEI_3.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Estanyeres_SPEI_12.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Plan_deth_Horn_SPEI_3.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Plan_deth_Horn_SPEI_12.png
📦 Guardado: M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\boxplots\boxplot_Pla_de_Sartari_SPEI_3.png
📦 Guardado: M:\material_isa\estadisticos\_Turbe

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

...

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

In [104]:
#----------------------------------------------------
# boxplots de NDDI agrupado por rangos de SPEI
#----------------------------------------------------

import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# CONFIGURACIÓN
# -------------------------------

input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\NDDI_filtr"
output_dir = r"M:\material_isa\estadisticos\_Turberas\SPEI_vs_NDDI\scatter plot"
os.makedirs(output_dir, exist_ok=True)

# Colores personalizados por turbera
colores = {
    "BassaNera": "#1f77b4",
    "ClotsdeRialba": "#ff7f0e",
    "Estanyeres": "#2ca02c",
    "PladeSartari": "#9467bd",
    "PlandethHorn": "#d62728",
    "PletadeMolieres": "#8c564b"
}

# -------------------------------
# CARGAR DATOS
# -------------------------------

# Diccionarios para acumular los datos de todas las turberas
datos_spei3 = []
datos_spei12 = []

for archivo in os.listdir(input_dir):
    if archivo.endswith(".csv"):
        ruta = os.path.join(input_dir, archivo)
        nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")
        nombre_color = nombre.replace("_", "")

        try:
            df = pd.read_csv(ruta, sep=";", encoding="latin1")
            if "SPEI_3" in df.columns and "SPEI_12" in df.columns and "NDDI" in df.columns:
                df = df[["SPEI_3", "SPEI_12", "NDDI"]].dropna()
                df["TURBERA"] = nombre_color
                datos_spei3.append(df[["SPEI_3", "NDDI", "TURBERA"]])
                datos_spei12.append(df[["SPEI_12", "NDDI", "TURBERA"]])
        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

# -------------------------------
# GRAFICAR DISPERSIÓN
# -------------------------------

def graficar(datos, spei_col, archivo_salida):
    if not datos:
        print(f"⚠️ Sin datos para {spei_col}")
        return

    df_todo = pd.concat(datos, ignore_index=True)

    plt.figure(figsize=(9, 6))
    for turbera in df_todo["TURBERA"].unique():
        df_turb = df_todo[df_todo["TURBERA"] == turbera]
        plt.scatter(
            df_turb[spei_col],
            df_turb["NDDI"],
            label=turbera.replace("de", " de").replace("la", " la"),
            color=colores.get(turbera, "gray"),
            alpha=0.6,
            edgecolor='k'
        )

    plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.xlabel(spei_col)
    plt.ylabel("NDDI")
    plt.title(f"{spei_col} vs NDDI – Todas las turberas")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"scatter_todas_{spei_col}.png"), dpi=150)
    plt.close()
    print(f"✅ Gráfico guardado: scatter_todas_{spei_col}.png")

# Ejecutar gráficos
graficar(datos_spei3, "SPEI_3", "scatter_todas_SPEI_3.png")
graficar(datos_spei12, "SPEI_12", "scatter_todas_SPEI_12.png")

print("\n🎯 Gráficos de dispersión por SPEI generados.")


✅ Gráfico guardado: scatter_todas_SPEI_3.png
✅ Gráfico guardado: scatter_todas_SPEI_12.png

🎯 Gráficos de dispersión por SPEI generados.


In [1]:
#------------------------------
# 5 indices pero en subplot
#---------------------------------


import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re

# -------------------------------
# CONFIGURACIÓN
# -------------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada"
input_dir_nddi = os.path.join(input_dir, "NDDI_filtr")
output_dir = os.path.join(input_dir, "Por_turbera")
snow_txt_path = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"
date_format = "%Y-%m-%d"

os.makedirs(output_dir, exist_ok=True)

# -------------------------------
# FUNCIONES
# -------------------------------

def cargar_periodos_nieve(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    torberas_periodos = {}
    bloques = contenido.split('✅ Procesado correctamente: ')[1:]
    for bloque in bloques:
        lineas = bloque.split('\n')
        nombre_raw = lineas[0].strip().replace('.csv', '')
        nombre = nombre_raw.replace('_periodo_nieve', '')
        periodos = []
        for linea in lineas:
            match = re.search(r'Snow in: (\d{2}/\d{2}/\d{4}) \| Snow off: (\d{2}/\d{2}/\d{4})', linea)
            if match:
                snow_in = datetime.strptime(match.group(1), '%d/%m/%Y')
                snow_off = datetime.strptime(match.group(2), '%d/%m/%Y')
                periodos.append((snow_in, snow_off))
        torberas_periodos[nombre] = periodos
    return torberas_periodos

# -------------------------------
# EJECUCIÓN
# -------------------------------
periodos_nieve = cargar_periodos_nieve(snow_txt_path)

for archivo in os.listdir(input_dir):
    if not archivo.endswith(".csv") or "NDDI" in archivo:
        continue

    nombre = archivo.replace("_sin_periodo_nieve_filtr_3_12.csv", "")
    ruta_base = os.path.join(input_dir, archivo)
    ruta_nddi = os.path.join(input_dir_nddi, archivo)

    try:
        df = pd.read_csv(ruta_base, sep=";", encoding="latin1")
        df_nddi = pd.read_csv(ruta_nddi, sep=";", encoding="latin1") if os.path.exists(ruta_nddi) else pd.DataFrame()

        df['FECHA'] = df['FECHA'].astype(str).str.strip()
        df['FECHA_dt'] = pd.to_datetime(df['FECHA'], format=date_format, errors='coerce')
        if not df_nddi.empty:
            df_nddi['FECHA'] = df_nddi['FECHA'].astype(str).str.strip()
            df_nddi['FECHA_dt'] = pd.to_datetime(df_nddi['FECHA'], format=date_format, errors='coerce')

        # -------------------------------
        # Preparar la figura con subplots
        # -------------------------------
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

        colores = {
            'NDVI': 'green',
            'NDWI_V': 'blue',
            'NDDI': 'orange',
            'SPEI_3': 'red',
            'SPEI_12': 'black'
        }

        marcadores = {
            'NDVI': 'o',
            'NDWI_V': 'o',
            'NDDI': 'o',
            'SPEI_3': 's',
            'SPEI_12': 's'
        }

        # -------------------------------
        # Plot arriba: NDVI, NDWI_v, NDDI
        # -------------------------------
        for indice in ['NDVI', 'NDWI_V']:
            if indice in df.columns:
                datos = df.dropna(subset=['FECHA_dt', indice])
                ax1.scatter(datos['FECHA_dt'], datos[indice],
                            facecolors='none',
                            edgecolors=colores[indice],
                            marker=marcadores[indice],
                            label=indice,
                            s=20)

        if not df_nddi.empty and 'NDDI' in df_nddi.columns:
            ax1.scatter(df_nddi['FECHA_dt'], df_nddi['NDDI'],
                        facecolors='none',
                        edgecolors=colores['NDDI'],
                        marker='o',
                        label='NDDI',
                        s=20)

        ax1.set_ylabel("Índices de vegetación")
        ax1.grid(True)
        ax1.legend(loc="upper left")
        ax1.set_title(f"Series Temporales – {nombre.replace('_', ' ')}")

        # -------------------------------
        # Plot abajo: SPEI_3 y SPEI_12
        # -------------------------------
        for indice in ['SPEI_3', 'SPEI_12']:
            if indice in df.columns:
                datos = df.dropna(subset=['FECHA_dt', indice])
                ax2.scatter(datos['FECHA_dt'], datos[indice],
                            facecolors='none',
                            edgecolors=colores[indice],
                            marker=marcadores[indice],
                            label=indice,
                            s=5,
                            alpha=0.7)

        ax2.set_ylabel("SPEI")
        ax2.set_xlabel("Fecha")
        ax2.grid(True)
        ax2.legend(loc="upper left")

        # -------------------------------
        # Añadir periodos de nieve en ambos
        # -------------------------------
        for ax in [ax1, ax2]:
            for snow_in, snow_off in periodos_nieve.get(nombre, []):
                ax.axvspan(snow_in, snow_off, color='gray', alpha=0.2)

        # Leyenda personalizada para SNOW
        ax1.text(1.01, 1.02, "Período SNOW", transform=ax1.transAxes,
                 fontsize=9, ha='right', color='gray',
                 bbox=dict(boxstyle="round,pad=0.2", facecolor='lightgray', edgecolor='none', alpha=0.5))

        # -------------------------------
        # Guardar figura
        # -------------------------------
        plt.tight_layout()
        salida_path = os.path.join(output_dir, f"{nombre}_series_temporales_subplot.png")
        plt.savefig(salida_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✅ Subplot generado: {salida_path}")

    except Exception as e:
        print(f"❌ Error procesando {archivo}: {e}")


✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Bassa_Nera_series_temporales_subplot.png
✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Clots_de_Rialba_series_temporales_subplot.png
✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Estanyeres_series_temporales_subplot.png
✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Plan_deth_Horn_series_temporales_subplot.png
✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Pla_de_Sartari_series_temporales_subplot.png
✅ Subplot generado: M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\Base de datos_Indices_filtrada\Por_turbera\Pleta_de_Molieres_series_t